# E4：320×320 输入分辨率对照实验

已内嵌代码、配置、冻结切分与测试，无需从 GitHub 下载源码，也不在本机运行。
先跑「环境准备」与「验证」两格，核对单变量断言、320×320 数据形状和显存占用，再启动训练。

## 运行前设置
- GPU：选择 T4，与团队基线 `baseline-v1` 保持一致。
- 挂载数据：`ozguraslank/brain-stroke-ct-dataset`。
- **本版为联网版**：Settings 里必须打开 Internet，且只需要挂载一个数据数据集。
- torch 由 pip 安装并锁死 `2.10.0+cu128`，与基线同栈；版本不一致时直接报错停止。
- 实验变量只有 `data.image_size`：256 → 320。模型、损失、优化器、batch size 8、15 轮、
  种子 42、阈值 0.5、冻结切分和统一评估脚本全部与基线一致。
- 图像按 BILINEAR、掩码按 NEAREST 从同一张原图同步缩放到 320×320，不存在图像/掩码错位。
- 显存不足时不允许改 batch size；应由基线负责人评估是否需要梯度累积（`scripts/train.py` 目前没有实现）。
- 推理延迟格输出纯前向与端到端两种口径的 ms/图，写入 `latency_benchmark.json`。
- 正式训练与最终评估默认关闭，Run All 不会直接训练。

In [ ]:
from pathlib import Path
import base64
import importlib.metadata
import io
import os
import subprocess
import sys
import tarfile
import uuid

repo = Path('/kaggle/working') / ('brain-stroke-e4-' + uuid.uuid4().hex[:12])
repo.mkdir(parents=True)
payload = '...'
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(payload)), mode='r:gz') as archive:
    try:
        archive.extractall(repo, filter='data')
    except TypeError:  # Python < 3.12 没有 filter 参数
        archive.extractall(repo)

candidates = [
    path for path in Path('/kaggle/input').rglob('Brain_Stroke_CT_Dataset')
    if all((path / name).is_dir() for name in ('Bleeding', 'Ischemia', 'Normal', 'External_Test'))
]
if not candidates:
    raise FileNotFoundError(
        '没有找到 Brain_Stroke_CT_Dataset。请先在 Input 挂载团队自建的私有数据集，'
        '其内部需包含 Brain_Stroke_CT_Dataset/Bleeding|Ischemia|Normal|External_Test。'
    )
data_root = candidates[0]
os.environ['BSC_DATA_ROOT'] = str(data_root)
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['MPLBACKEND'] = 'Agg'
os.environ['REQUIRE_CUDA'] = '1'
config_path = 'configs/experiments/E4_res320.yaml'
print('仓库:', repo)
print('数据:', data_root)
print('配置:', config_path)


def pip_install(arguments):
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check', *arguments],
        check=True,
    )


def installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


torch_version = installed('torch')
print('预装 torch:', torch_version)
if torch_version != '2.10.0+cu128':
    pip_install(['torch==2.10.0+cu128', '--index-url', 'https://download.pytorch.org/whl/cu128'])
    torch_version = installed('torch')
    print('安装后 torch:', torch_version)
if torch_version != '2.10.0+cu128':
    raise RuntimeError('torch 未锁定到 2.10.0+cu128，与基线不同栈，停止运行')

missing = [spec for spec in ('numpy', 'pandas', 'Pillow', 'matplotlib', 'tqdm', 'PyYAML') if installed(spec) is None]
if missing:
    pip_install(missing)
    print('补装依赖:', missing)

subprocess.run([
    sys.executable, '-c',
    "import torch; assert torch.__version__ == '2.10.0+cu128', torch.__version__; "
    "assert torch.cuda.is_available(), 'CUDA unavailable'; "
    "print('torch 2.10.0+cu128 + CUDA 就绪:', torch.cuda.get_device_name(0))"
], check=True)

In [ ]:
import hashlib
import subprocess
import sys
import textwrap
from pathlib import Path

E4_VERIFIED = False
EXPECTED_SPLIT_SHA256 = '4853e4fe55dac627ae4a5daba918aa00f46feb156932e909cbce5802742a7052'

actual_split = hashlib.sha256((repo / 'data/splits/main_split.json').read_bytes()).hexdigest()
assert actual_split == EXPECTED_SPLIT_SHA256, '冻结切分校验失败'
print('冻结切分校验通过:', actual_split)

def run_checked(arguments, log_name):
    result = subprocess.run(
        [sys.executable, *arguments], cwd=repo, env=os.environ.copy(),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    (repo / log_name).write_text(result.stdout, encoding='utf-8')
    print(result.stdout)
    result.check_returncode()

run_checked(['-m', 'unittest', 'discover', '-s', 'tests', '-v'], 'e4_tests.log')

smoke = textwrap.dedent("""
    import torch
    import yaml
    from torch.utils.data import DataLoader

    from src.data import dataset_for_split, ensure_split
    from src.losses import build_loss
    from src.models import build_model
    from src.utils import resolve_data_root, seed_everything

    with open('configs/baseline.yaml', encoding='utf-8') as stream:
        baseline = yaml.safe_load(stream)
    with open('configs/experiments/E4_res320.yaml', encoding='utf-8') as stream:
        config = yaml.safe_load(stream)

    # 单变量断言：除 image_size 及命名/输出目录外，其余必须与基线逐项相同
    assert config['data']['image_size'] == 320, config['data']['image_size']
    assert baseline['data']['image_size'] == 256, baseline['data']['image_size']
    for key in ('model', 'loss', 'train', 'evaluation'):
        assert config[key] == baseline[key], key
    for key, value in baseline['data'].items():
        if key in ('image_size', 'generate_split_if_missing'):
            continue
        assert config['data'][key] == value, key
    assert config['data']['generate_split_if_missing'] is False
    assert config['project']['seed'] == baseline['project']['seed'] == 42
    assert config['project']['output_dir'] == 'reports/experiments/E4_res320'
    print('单变量校验通过：仅 image_size 256 -> 320')

    assert torch.cuda.is_available(), 'CUDA 不可用'
    total_gib = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    print('GPU:', torch.cuda.get_device_name(0), f'{total_gib:.1f} GiB')

    seed_everything(config['project']['seed'])
    root = resolve_data_root(config['data']['root'])
    split = ensure_split(
        root=root, split_path='data/splits/main_split.json', seed=42,
        generate_if_missing=False, train_fraction=0.70,
        val_fraction=0.15, test_fraction=0.15,
    )
    print('切分:', split['version'], split['counts'])

    dataset = dataset_for_split(
        root=root, split_payload=split, split_name='train',
        image_size=320, mask_mode='red', training=False,
    )
    item = dataset[0]
    assert tuple(item['image'].shape) == (1, 320, 320), item['image'].shape
    assert tuple(item['mask'].shape) == (1, 320, 320), item['mask'].shape
    print('数据样本:', item['image_id'], tuple(item['image'].shape), '| 掩码与图像同步缩放')

    loader = DataLoader(dataset, batch_size=8, shuffle=False, num_workers=0)
    batch = next(iter(loader))
    images = batch['image'].cuda()
    masks = batch['mask'].cuda()

    model = build_model(config['model']).cuda()
    criterion = build_loss(config['loss'])
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(config['train']['learning_rate']))
    scaler = torch.amp.GradScaler('cuda', enabled=bool(config['train']['amp']))

    torch.cuda.reset_peak_memory_stats()
    with torch.autocast(device_type='cuda', enabled=bool(config['train']['amp'])):
        loss = criterion(model(images), masks)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    peak_train = torch.cuda.max_memory_allocated() / 1024 ** 3
    reserved_train = torch.cuda.max_memory_reserved() / 1024 ** 3
    assert torch.isfinite(loss), float(loss)
    assert all(
        torch.isfinite(parameter.grad).all()
        for parameter in model.parameters() if parameter.grad is not None
    )
    print('训练步通过 | batch=8 @320x320 | loss=', float(loss.detach()))
    print(f'训练峰值显存 allocated={peak_train:.2f} GiB '
          f'reserved={reserved_train:.2f} GiB / total={total_gib:.2f} GiB')

    torch.cuda.reset_peak_memory_stats()
    model.eval()
    with torch.inference_mode(), torch.autocast(device_type='cuda', enabled=bool(config['train']['amp'])):
        logits = model(torch.randn(16, 1, 320, 320, device='cuda'))
    assert tuple(logits.shape) == (16, 1, 320, 320), logits.shape
    peak_eval = torch.cuda.max_memory_allocated() / 1024 ** 3
    print(f'评估推理通过 | batch=16 @320x320 | 峰值 allocated={peak_eval:.2f} GiB')
    if peak_train > 0.9 * total_gib:
        raise RuntimeError('训练峰值显存超过总显存的 90%，正式训练有 OOM 风险')
    print('显存余量检查通过，可以启动正式训练。')
""")
run_checked(['-c', smoke], 'e4_cuda_smoke.log')
E4_VERIFIED = True
print('验证完成：本轮没有启动训练。确认显存与断言结果后再切正式版 Notebook。')

In [ ]:
# 默认关闭：先核对上一格的测试、单变量与显存结果，再决定是否启动完整训练。
RUN_TRAINING = False
if RUN_TRAINING:
    assert E4_VERIFIED, '先完成验证格'
    subprocess.run([
        sys.executable, '-c',
        "import json, platform, torch; from pathlib import Path; "
        "p = Path('reports/experiments/E4_res320'); p.mkdir(parents=True, exist_ok=True); "
        "(p / 'environment.json').write_text(json.dumps({'python': platform.python_version(), "
        "'torch': torch.__version__, 'cuda': torch.version.cuda, "
        "'cudnn': torch.backends.cudnn.version(), 'gpu': torch.cuda.get_device_name(0)}, "
        "indent=2), encoding='utf-8')"
    ], cwd=repo, check=True, env=os.environ.copy())
    subprocess.run(
        [sys.executable, 'scripts/train.py', '--config', config_path],
        cwd=repo, check=True, env=os.environ.copy(),
    )
else:
    print('正式训练未启动。')

In [ ]:
import subprocess
import sys

bench_path = repo / 'e4_latency_bench.py'
bench_path.write_text("import os\nfrom pathlib import Path\n\n# 由 Notebook 写入仓库根后执行：cwd 即仓库根，因此 src 可直接导入\nrepo = Path.cwd()\nconfig_path = 'configs/experiments/E4_res320.yaml'\n\nimport json\nimport time\nfrom pathlib import Path\n\nimport torch\nimport yaml\n\nfrom src.data import dataset_for_split, ensure_split\nfrom src.models import build_model\nfrom src.utils import resolve_data_root\n\nconfig = yaml.safe_load((repo / config_path).read_text(encoding='utf-8'))\noutput_dir = repo / config['project']['output_dir']\noutput_dir.mkdir(parents=True, exist_ok=True)\n\ncandidates = [output_dir / 'best.pt'] + sorted(Path('/kaggle/input').rglob('best.pt'))\ncheckpoint = next((path for path in candidates if path.is_file()), None)\n\nmodel = build_model(config['model'], pretrained=False).cuda()\nweight_info = {'source': 'random_init', 'path': None, 'checkpoint_epoch': None}\nif checkpoint is not None:\n    state = torch.load(checkpoint, map_location='cuda', weights_only=False)\n    model.load_state_dict(state['state_dict'])\n    weight_info = {\n        'source': 'best.pt',\n        'path': str(checkpoint),\n        'checkpoint_epoch': int(state.get('epoch', -1)),\n    }\n    print('已加载训练权重:', checkpoint, '| epoch =', weight_info['checkpoint_epoch'])\nelse:\n    print(\n        '未找到 best.pt：改用同结构随机初始化权重。前向延迟只取决于网络结构与输入尺寸，'\n        '与权重数值无关，因此该结果仍可用于报告；若需要记录 epoch，请把结果 zip 挂到 /kaggle/input。'\n    )\nmodel.eval()\namp = bool(config['train']['amp'])\n\ndef forward_latency(size, batch, repeats, warmup=15):\n    sample = torch.randn(batch, 1, size, size, device='cuda')\n    with torch.inference_mode(), torch.autocast(device_type='cuda', enabled=amp):\n        for _ in range(warmup):\n            model(sample)\n        torch.cuda.synchronize()\n        started = time.perf_counter()\n        for _ in range(repeats):\n            model(sample)\n        torch.cuda.synchronize()\n    return (time.perf_counter() - started) / repeats / batch * 1000.0\n\nmeasurements = []\nfor size in (320, 256):\n    for batch, repeats in ((1, 100), (16, 20)):\n        milliseconds = forward_latency(size, batch, repeats)\n        measurements.append({\n            'kind': 'forward_only',\n            'image_size': size,\n            'batch': batch,\n            'ms_per_image': round(milliseconds, 3),\n            'images_per_second': round(1000.0 / milliseconds, 2),\n        })\n        print(f'纯前向 {size}x{size} batch={batch}: {milliseconds:.2f} ms/图 '\n              f'({1000.0 / milliseconds:.1f} 图/秒)')\n\n# 端到端：含 PNG 解码 + resize + 前向，batch=1，模拟逐例出报告的场景\ntry:\n    root = resolve_data_root(config['data']['root'])\n    split = ensure_split(\n        root=root, split_path=config['data']['split_path'], seed=int(config['project']['seed']),\n        generate_if_missing=False, train_fraction=float(config['data']['train_fraction']),\n        val_fraction=float(config['data']['val_fraction']), test_fraction=float(config['data']['test_fraction']),\n    )\n    dataset = dataset_for_split(\n        root=root, split_payload=split, split_name='external_test',\n        image_size=int(config['data']['image_size']), mask_mode=str(config['data']['mask_mode']),\n        training=False,\n    )\n    sample_count = min(30, len(dataset))\n    for _ in range(3):\n        with torch.inference_mode(), torch.autocast(device_type='cuda', enabled=amp):\n            model(dataset[0]['image'].unsqueeze(0).cuda())\n    torch.cuda.synchronize()\n    started = time.perf_counter()\n    with torch.inference_mode(), torch.autocast(device_type='cuda', enabled=amp):\n        for index in range(sample_count):\n            image = dataset[index]['image'].unsqueeze(0).cuda()\n            model(image)\n    torch.cuda.synchronize()\n    end_to_end_ms = (time.perf_counter() - started) / sample_count * 1000.0\n    measurements.append({\n        'kind': 'end_to_end_decode_resize_forward',\n        'image_size': int(config['data']['image_size']),\n        'batch': 1,\n        'samples': sample_count,\n        'ms_per_image': round(end_to_end_ms, 3),\n    })\n    print(f'端到端（解码+resize+前向）{sample_count} 例: {end_to_end_ms:.2f} ms/图')\nexcept Exception as error:  # 数据未挂载时不阻塞纯前向基准\n    print('端到端测速跳过（数据不可用）:', repr(error))\n\nbenchmark = {\n    'experiment': config['project']['name'],\n    'gpu': torch.cuda.get_device_name(0),\n    'torch': torch.__version__,\n    'amp': amp,\n    'weights': weight_info,\n    'note': 'forward_only 不含 PNG 解码与 resize；end_to_end 为该机的实测上限口径',\n    'measurements': measurements,\n}\n(output_dir / 'latency_benchmark.json').write_text(\n    json.dumps(benchmark, ensure_ascii=False, indent=2) + '\\n', encoding='utf-8'\n)\nprint('已写出:', output_dir / 'latency_benchmark.json')", encoding='utf-8')
finished = subprocess.run(
    [sys.executable, str(bench_path)], cwd=repo, env=os.environ.copy(),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
(repo / 'e4_latency_bench.log').write_text(finished.stdout, encoding='utf-8')
print(finished.stdout)
finished.check_returncode()
print('延迟基准文件:', repo / 'reports/experiments/E4_res320/latency_benchmark.json')

In [ ]:
# 仅在最佳权重生成后开启；不要用测试集或 External_Test 调参、选轮次或改阈值。
RUN_FINAL_EVALUATION = False
if RUN_FINAL_EVALUATION:
    assert E4_VERIFIED, '先完成验证格'
    assert (repo / 'reports/experiments/E4_res320/best.pt').is_file(), '缺少 best.pt'
    for entry in ('evaluate.py', 'infer.py'):
        subprocess.run(
            [sys.executable, 'scripts/' + entry, '--config', config_path],
            cwd=repo, check=True, env=os.environ.copy(),
        )
else:
    print('内部测试与 External_Test 评估未启动。')

In [ ]:
# 打包 E4 产出（可能只有验证日志），供下载保存。
import zipfile
bundle = Path('/kaggle/working') / (repo.name + '-e4-results.zip')
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in repo.rglob('*'):
        if path.is_file() and (
            'E4_res320' in path.parts
            or path.name in ('e4_tests.log', 'e4_cuda_smoke.log', 'e4_latency_bench.log')
        ):
            archive.write(path, path.relative_to(repo))
print('结果包：', bundle)
print('权重可保留在下载包中，但不要提交到 Git。')